# Progression constraints from cross-sectional data with MHN

A cross-sectional cohort tells you *which* alterations a patient carries, not the order they arrived
in. **[MHN](https://github.com/spang-lab/LearnMHN)** (Mutual Hazard Networks, Schill et al. 2020) is
built for exactly that: it fits a network of promoting and inhibiting effects between events from a
binary patients x events matrix — the shape a bulk or panel-sequencing cohort actually has.

This runs the real `mhn` package **in this notebook's own kernel** (`Python (iscc-mhn)`), so the
fitted model object is here to inspect and its own plotting works.

The cohort is the one generated for
[the TreeMHN example](tool_treemhn_R.ipynb) — the *same* patients, projected down to presence/absence.
See [The analysis dataset and its ground truth](analysis_ground_truth.ipynb) for how it was made:

| file | what it is | who sees it |
|---|---|---|
| `X_presence.csv` | patients x events, binary | the tool |
| `truth_network.json` | the planted precedence DAG | **scoring only** |

**Why this is measurable.** Real cohorts have no answer key. Here every patient was grown under a
known DAG stating which events must precede which, so a recovered dependency can be checked against
the one that was planted.

In [ ]:
import json, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from mhn.optimizers import cMHNOptimizer

DATA = os.path.join("..", "analysis_data", "treemhn")
assert os.path.isdir(DATA), (
    f"dataset not found at {DATA}\n"
    "  generate it first:  python validation/make_analysis_data.py --only treemhn")

X     = pd.read_csv(os.path.join(DATA, "X_presence.csv"), index_col=0)
truth = json.load(open(os.path.join(DATA, "truth_network.json")))
dag   = truth["true_dag_edges"]            # rows: (parent, child), 0-based event ids
events = list(X.columns)

print(f"cohort: {X.shape[0]} patients, {X.shape[1]} events")
print(f"gating: {truth['dependency_params']['gating_mode']}   "
      f"planted precedence constraints: {len(dag)}")
for p, c in dag:
    print(f"   {events[p]}  must precede  {events[c]}")
print("\nevent frequency across the cohort:")
print(X.mean(0).round(2).to_string())

## Fit

`cMHNOptimizer` is the package's own interface. MHN is regularised, and the penalty matters: too weak
and every pair picks up a non-zero effect, so the planted edges drown in false positives. The package
prescribes choosing it by cross-validation, which is what `lambda_from_cv` does.

In [ ]:
opt = cMHNOptimizer()
opt.load_data_matrix(X)                        # a DataFrame, so the model keeps the event names

lam = opt.lambda_from_cv(nfolds=5, show_progressbar=False)
model = opt.train(lam=lam, maxit=5000)

print(f"lambda chosen by 5-fold CV: {lam:.5f}")
print(f"fitted {type(model).__name__} over events {model.events}")

## MHN's own figure: the Θ matrix

`plot()` is the package's own figure, the same one in the MHN papers. The **diagonal** holds each
event's baseline rate; an **off-diagonal** `Θ[i, j]` is the multiplicative effect of event *j* on the
rate of event *i* — red promotes, blue inhibits.

A planted "*p* must precede *c*" should therefore surface as a large positive entry in **row *c*,
column *p***.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.6))
model.plot(ax=ax, annot=0.1)
ax.set_title("MHN Θ estimated from binary presence")
plt.show()

## Scoring against the planted constraints

The question is not whether Θ looks plausible — it is whether the planted pairs are the *top-ranked*
promoting entries out of every off-diagonal.

In [ ]:
Theta = np.asarray(model.log_theta)
n = Theta.shape[0]

off = [(i, j) for i in range(n) for j in range(n) if i != j]
order = sorted(off, key=lambda ij: -Theta[ij])

print(f"recovery of the {len(dag)} planted edges "
      f"(rank among {len(off)} off-diagonals, 1 = strongest)\n")
ranks = []
for p, c in dag:
    r = order.index((c, p)) + 1
    ranks.append(r)
    print(f"  {events[p]} -> {events[c]} :  Theta = {Theta[c, p]:+.3f}   rank {r} of {len(off)}")

print("\nstrongest off-diagonals overall:")
for i, j in order[:4]:
    planted = " <- planted" if [j, i] in [list(e) for e in dag] else ""
    print(f"   {events[j]} -> {events[i]} : {Theta[i, j]:+.3f}{planted}")

## MHN's own figure: the likeliest progression orders

Given a fitted network, MHN can say which chronological order most likely produced each observed
patient state. `plot_order_tree` aggregates those over the cohort: every root-to-leaf path is a
progression trajectory, and line width scales with how many patients took it.

This is inference, not observation — the input matrix carries no order at all. It is the closest
cross-sectional analogue of TreeMHN's `plot_pathways`.

In [ ]:
states = X.values.astype(np.int32)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
model.plot_order_tree(states=states, ax=ax, min_number_of_occurrence=2, legend=True)
ax.set_title("likeliest progression orders inferred by MHN")
plt.show()

## What to take from it

MHN fits `iscc` cohorts out of the box, in the shape real cross-sectional data comes in, and its
estimate is checkable because the generating constraints were recorded.

The comparison worth making is with [the same cohort read as trees](tool_treemhn_R.ipynb). TreeMHN
sees the order in which each patient acquired its events; MHN sees only the endpoint. Whatever
structure survives the binary projection is what a cross-sectional design can recover — and the gap
between the two rankings is the value of the extra resolution, measured rather than asserted.

That is a statement about *observables*, not about either tool: both are fitted here to the same
patients, generated under the same known DAG.